# Noisy emulation with Qiskit-compiled circuits on Amazon Braket

In [this notebook](https://github.com/amazon-braket/amazon-braket-examples/blob/main/examples/braket_features/Device_emulation/01_Local_Emulation_for_Verbatim_Circuits_on_Amazon_Braket.ipynb), we introduced how to emulate verbatim circuits locally based on device calibration data. In this notebook, we show how to combine the [Qiskit-Braket provider](https://github.com/qiskit-community/qiskit-braket-provider) compilation pipeline with the local emulator to emulate non-verbatim circuits for a targetted QPU.

The workflow is:
1. Build a circuit using the **Braket SDK** with arbitrary gates.
2. Convert to Qiskit, then use the Qiskit transpiler to compile it to the **native gate set** of a target QPU.
3. Convert the transpiled circuit back to a Braket verbatim circuit.
4. Run the verbatim circuit on the Braket **local emulator** with a realistic noise model derived from device calibration data.

We will demonstrate the workflow using Rigetti [Cepheus-1-108Q](https://aws.amazon.com/braket/quantum-computers/rigetti/).

In [34]:
from braket.tracking import Tracker

t = Tracker().start()

## Choose a target QPU

Change `DEVICE_ARN` to retarget the entire notebook to a different QPU.

In [35]:
from braket.aws import AwsDevice

DEVICE_ARN = "arn:aws:braket:us-west-1::device/qpu/rigetti/Cepheus-1-108Q"
# DEVICE_ARN = "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet"
# DEVICE_ARN = "arn:aws:braket:eu-north-1::device/qpu/iqm/Emerald"
# DEVICE_ARN = "arn:aws:braket:us-east-1::device/qpu/ionq/Forte-1"
# DEVICE_ARN = "arn:aws:braket:us-east-1::device/qpu/ionq/Forte-Enterprise-1"
# DEVICE_ARN = "arn:aws:braket:eu-north-1::device/qpu/aqt/Ibex-Q1"

device = AwsDevice(DEVICE_ARN)
print(f"Device: {device.name}")
print(f"Native gates: {device.properties.paradigm.nativeGateSet}")
print(f"Qubits: {device.properties.paradigm.qubitCount}")

Device: Cepheus-1-108Q
Native gates: ['rx', 'rz', 'cz', 'barrier']
Qubits: 108


## Build a high-level circuit with the Braket SDK

We create a 4-qubit GHZ circuit using standard gates (`H`, `CNot`). These are **not** native to the target QPU and must be compiled before verbatim execution.

In [36]:
from braket.circuits import Circuit

NUM_QUBITS = 4

braket_circuit = Circuit()
braket_circuit.h(0)
for i in range(NUM_QUBITS - 1):
    braket_circuit.cnot(i, i + 1)

print("Original Braket circuit:")
print(braket_circuit)

Original Braket circuit:
T  : │  0  │  1  │  2  │  3  │
      ┌───┐                   
q0 : ─┤ H ├───●───────────────
      └───┘   │               
            ┌─┴─┐             
q1 : ───────┤ X ├───●─────────
            └───┘   │         
                  ┌─┴─┐       
q2 : ─────────────┤ X ├───●───
                  └───┘   │   
                        ┌─┴─┐ 
q3 : ───────────────────┤ X ├─
                        └───┘ 
T  : │  0  │  1  │  2  │  3  │


## Convert to Qiskit and transpile to native gates

We use `from_braket()` to convert the Braket circuit to a Qiskit `QuantumCircuit`, then transpile it using a `BraketLocalBackend` that is aware of the target device's native gates and connectivity. We note that `BraketLocalBackend` performs transpilation **locally** — it does not submit any tasks to AWS.

In [37]:
from qiskit import transpile
from qiskit_braket_provider import BraketProvider, to_qiskit, to_braket

# Braket -> Qiskit
qiskit_circuit = to_qiskit(braket_circuit)

# Transpile to native gates
local_backend = BraketProvider().get_backend(device.name)
# Explicitly set basis_gates to the device's native gate set
native_gates = [g.lower() for g in device.properties.paradigm.nativeGateSet if g.lower() not in ['barrier', 'cc_prx', 'measure_ff']]
transpiled_qc = transpile(qiskit_circuit, backend=local_backend, basis_gates=native_gates, optimization_level=1)

print("Transpiled Qiskit verbatim circuit:")
print(transpiled_qc)

Transpiled Qiskit verbatim circuit:
global phase: 3π/2
         ┌─────────┐┌─────────┐┌─────────┐                                    »
q_0 -> 0 ┤ Rz(π/2) ├┤ Rx(π/2) ├┤ Rz(π/2) ├─■──────────────────────────────────»
         ├─────────┤├─────────┤├─────────┤ │ ┌─────────┐┌─────────┐┌─────────┐»
q_1 -> 1 ┤ Rz(π/2) ├┤ Rx(π/2) ├┤ Rz(π/2) ├─■─┤ Rz(π/2) ├┤ Rx(π/2) ├┤ Rz(π/2) ├»
         ├─────────┤├─────────┤├─────────┤   └─────────┘└─────────┘└─────────┘»
q_2 -> 2 ┤ Rz(π/2) ├┤ Rx(π/2) ├┤ Rz(π/2) ├────────────────────────────────────»
         ├─────────┤├─────────┤├─────────┤                                    »
q_3 -> 3 ┤ Rz(π/2) ├┤ Rx(π/2) ├┤ Rz(π/2) ├────────────────────────────────────»
         └─────────┘└─────────┘└─────────┘                                    »
 meas: 4/═════════════════════════════════════════════════════════════════════»
                                                                              »
«                                                                

## Convert back to a Braket verbatim circuit

We use `to_braket()` to convert the transpiled circuit back to a Braket `Circuit`, then wrap it in a **verbatim box** so the Braket compiler will not modify it.

In [38]:
transpiled_circ_no_meas = transpiled_qc.remove_final_measurements(inplace=False)
transpiled_circ_no_meas.global_phase = 0
compiled_braket = to_braket(transpiled_circ_no_meas)
verbatim_circuit = Circuit().add_verbatim_box(compiled_braket)

print("Braket verbatim circuit:")
print(verbatim_circuit)

Braket verbatim circuit:
T  : │        0        │     1      │     2      │     3      │  4  │     5      │     6      │     7      │  8  │     9      │     10     │     11     │ 12  │     13     │     14     │     15     │      16       │
                        ┌──────────┐ ┌──────────┐ ┌──────────┐                                                                                                                                                        
q0 : ───StartVerbatim───┤ Rz(1.57) ├─┤ Rx(1.57) ├─┤ Rz(1.57) ├───●──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────EndVerbatim───
              ║         └──────────┘ └──────────┘ └──────────┘   │                                                                                                                                           ║        
              ║         ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌─┴─┐ ┌──────────┐ ┌──────────┐ ┌──────────┐        

## Run noisy emulation on the local emulator

The local emulator is created from the target device via `device.emulator()`. It automatically loads the latest calibration data (gate fidelities, readout errors, T1/T2 times) to build a realistic noise model.

In [39]:
from collections import Counter

emulator = device.emulator()

SHOTS = 1000
result = emulator.run(verbatim_circuit, shots=SHOTS).result()
counts = Counter(result.measurement_counts)

print(f"Measurement counts ({SHOTS} shots):")
print(counts.most_common(10))

Measurement counts (1000 shots):
[('0111', 127), ('0000', 118), ('1111', 111), ('0001', 110), ('1000', 109), ('0110', 108), ('1001', 103), ('1110', 102), ('1100', 20), ('1101', 18)]


For an ideal GHZ state we expect only the all-zeros and all-ones bitstrings. The noisy emulator shows how device imperfections (gate errors, readout errors) spread probability to other bitstrings.

## Summary

In this notebook we demonstrated a complete workflow for:

1. **Building** a high-level quantum circuit with the Braket SDK.
2. **Compiling** it to a target QPU's native gate set using the Qiskit-Braket provider.
3. **Converting** the compiled circuit to a Braket verbatim circuit.
4. **Running** noisy emulation locally using the Braket device emulator.

This workflow is parameterized by `DEVICE_ARN`, so you can retarget it to any Braket QPU by changing a single variable.

In [40]:
print("Tracker summary:")
print(t.quantum_tasks_statistics())
print(f"Estimated cost: ${t.simulator_tasks_cost():.2f}")

Tracker summary:
{}
Estimated cost: $0.00
